## Run recon-any

2025/04/23: reused previous recon-all notebook to run recon-any  (adjusting the command as required). Scans were resampled  to every 3rd or 5th zslice to emulate lower quality clinical scans.

In [ ]:
from pathlib import Path
import pandas as pd
pd.set_option('display.max_rows', 200)

In [ ]:
pipeline_version = "any"
freesurfer_module_version = "freesurfer/8.0.0-beta"
ncpus = 10
zres_vals = [3,5]

In [ ]:
project_dir = Path("/data/ABCD_MBDU/ohbm2024/")
bids_dir = Path('/data/ABCD_DSST/ABCD/imaging_data/fast_track/rawdata/')
fsres_dir = project_dir / 'data/derivatives/freesurfer/'
swarm_cmd_dir = project_dir / 'swarm/swarm_commands'
swarm_log_dir = project_dir / 'swarm/swarm_logs'

balanced_scans = pd.read_csv(project_dir / 'code/balanced_scans.csv')

In [ ]:
balanced_scans['path'] = bids_dir/balanced_scans.filename

In [ ]:
assert balanced_scans.path.apply(lambda x: x.exists()).all()

In [ ]:
t1w_scans = balanced_scans.query("modality == 'T1w'").loc[:, ['participant_id', 'session_id', 'path']]
t1w_scans = t1w_scans.rename(columns={'path':'t1_path'})

t2w_scans = balanced_scans.query("modality == 'T2w'").loc[:, ['participant_id', 'session_id', 'path']]
t2w_scans = t2w_scans.rename(columns={'path':'t2_path'})

scans_to_run = t1w_scans.merge(t2w_scans, how='left', on=['participant_id', 'session_id'])
scans_to_run = scans_to_run.rename(columns={'participant_id':'subject', 'session_id':'session'}).groupby("session").head(250)

In [ ]:
scans_to_run.head()


# Run freesurfer recon-all on full res T1 + T2
(don't have to redo the original recon-all... already exists

# run recon-any clinical on full res t1 and t2

In [ ]:
!scancel 50405956

In [ ]:
# run_recon-any INPUT_SCAN SUBJECT_ID THREADS SIDE [SUBJECT_DIR]
cmds = []
for modality in "t1 t2".split():
    for _, row in scans_to_run.iterrows():
        fs_outdir = fsres_dir /f'recon-{pipeline_version}_{modality}'/ row.subject / row.session
        fs_outdir.mkdir(exist_ok=True, parents=True)
        if pipeline_version == 'any':
            cmd = f'export SUBJECTS_DIR=/lscratch/$SLURM_JOBID/fs_out; mkdir $SUBJECTS_DIR; '
            cmd += ' source ${FREESURFER_HOME}/SetUpFreeSurfer.sh; '
            cmd += f'run_recon-any {row[modality + "_path"].resolve()} {row.subject} {ncpus} both; '
            cmd += f'rsync -a /lscratch/$SLURM_JOBID/fs_out {fs_outdir.resolve()} ; '
            cmd += f'chown -R :ABCD_MBDU {fs_outdir.resolve()}; '
            cmd += f'chmod -R g+rw {fs_outdir.resolve()}; '
            cmd += ' rm -rf /lscratch/$SLURM_JOB_ID/*'
        cmds.append(cmd)



In [ ]:
cmds[-20:]

In [ ]:
test = False
run_name = f'abcd_recon_{pipeline_version}-leej3{"-test" if test else ""}'

swarm_cmd_file = swarm_cmd_dir / run_name
swarm_cmd_file.write_text('\n'.join(cmds[:2] if test else cmds))
swarm_exec = f"""swarm -f {swarm_cmd_file.resolve()} -g 100 -t {ncpus} --gres=lscratch:400 --module {freesurfer_module_version},fsl --time 12:00:00 --logdir {swarm_log_dir.resolve()} --job-name {run_name} --partition norm"""
print(swarm_exec)

In [ ]:
job_id = !{swarm_exec}
print(job_id)

# resample
(redo with recon-any at zres = 3 and zres = 5)

In [ ]:
## run_recon-any INPUT_SCAN SUBJECT_ID THREADS SIDE [SUBJECT_DIR]
assert pipeline_version == 'any'
zmcmds = {}
for zres in zres_vals:
    zmcmds[zres] = {}
    for modality in "t1 t2".split():
        zmcmds[zres][modality] = []
        for _, row in scans_to_run.iterrows():
            input_scan = row[modality + "_path"]
            fs_outdir = fsres_dir /f'recon-{pipeline_version}_{modality}_resample-{zres}'/ row.subject / row.session
            fs_outdir.mkdir(exist_ok=True, parents=True)
            resampled = f'/lscratch/$SLURM_JOBID/{input_scan.parts[-1]}'
            cmd = f'export SUBJECTS_DIR=/lscratch/$SLURM_JOBID/fs_out; mkdir $SUBJECTS_DIR; '
            cmd += ' source ${FREESURFER_HOME}/SetUpFreeSurfer.sh; '
            cmd += f'3dresample -rmode Cu -dxyz 1.0 1.0 {zres}.0 -prefix {resampled} -input {input_scan.resolve()} ; '
            cmd += f'run_recon-any {resampled} {row.subject} {ncpus} both; '
            cmd += f'rsync -a /lscratch/$SLURM_JOBID/fs_out {fs_outdir.resolve()} ; '
            cmd += f'chown -R :ABCD_MBDU {fs_outdir.resolve()}; '
            cmd += f'chmod -R g+rw {fs_outdir.resolve()}; '
            cmd += ' rm -rf /lscratch/$SLURM_JOB_ID/*'
            zmcmds[zres][modality].append(cmd)

In [ ]:
len(zmcmds)

In [ ]:
modality = 't1'
for zres in zres_vals:
    cmds = zmcmds[zres][modality]
    test = False
    run_name = f'leej3_abcd_recon_{pipeline_version}-withresample_zres-{zres}_modality-{modality}{"-test" if test else ""}'
    
    swarm_cmd_file = swarm_cmd_dir / run_name
    swarm_cmd_file.write_text('\n'.join(cmds[:2] if test else cmds))
    swarm_exec = f"""swarm -f {swarm_cmd_file.resolve()} -g 80 -t {ncpus} --gres=lscratch:400 --module afni,{freesurfer_module_version},fsl --time 12:00:00 --logdir {swarm_log_dir.resolve()} --job-name {run_name} --partition norm"""
    print(run_name)
    print(swarm_exec)

    job_id = !{swarm_exec}
    print(job_id)
    print()

In [ ]:
!sjobs

In [ ]:
!scancel 50406796

In [ ]:
!sjobs

## Results


In [ ]:
!tree {fsres_dir}/recon-any_t1/sub-NDARINV00HEV6HB

## t2 resampled was not run

In [ ]:
!ls -d {fsres_dir}/recon-any* |xargs -I % bash -c 'echo % count for aparc aseg is:  && find % -name aparc.a2009s+aseg.mgz|wc '

In [ ]:
!ls swarm/swarm_logs/leej3*t2*

In [ ]:
!ls swarm/swarm_logs/leej3*t1*|wc

In [ ]:
modality = 't2'
test = False
for zres in zres_vals:
    cmds = zmcmds[zres][modality]
    
    run_name = f'leej3_abcd_recon_{pipeline_version}-withresample_zres-{zres}_modality-{modality}{"-test" if test else ""}'
    
    swarm_cmd_file = swarm_cmd_dir / run_name
    swarm_cmd_file.write_text('\n'.join(cmds[:2] if test else cmds))
    swarm_exec = f"""swarm -f {swarm_cmd_file.resolve()} -g 80 -t {ncpus} --gres=lscratch:400 --module afni,{freesurfer_module_version},fsl --time 12:00:00 --logdir {swarm_log_dir.resolve()} --job-name {run_name} --partition norm"""
    print(run_name)
    print(swarm_exec)

    job_id = !{swarm_exec}
    print(job_id)
    print()

In [ ]:
modality = 't2'
test = False
for zres in zres_vals:
    cmds = zmcmds[zres][modality]
    
    run_name = f'leej3_abcd_recon_{pipeline_version}-withresample_zres-{zres}_modality-{modality}{"-test" if test else ""}'
    
    swarm_cmd_file = swarm_cmd_dir / run_name
    swarm_cmd_file.write_text('\n'.join(cmds[:2] if test else cmds[2:]))
    swarm_exec = f"""swarm -f {swarm_cmd_file.resolve()} -g 80 -t {ncpus} --gres=lscratch:400 --module afni,{freesurfer_module_version},fsl --time 12:00:00 --logdir {swarm_log_dir.resolve()} --job-name {run_name} --partition norm"""
    print(run_name)
    print(swarm_exec)

    job_id = !{swarm_exec}
    print(job_id)
    print()

In [ ]:
!sjobs

In [ ]:
!ls swarm/swarm_logs/leej3_abcd_recon_any-withresample_zres-3_modality-t2* | head

In [ ]:
!cat swarm/swarm_logs/leej3_abcd_recon_any-withresample_zres-3_modality-t2_51923006_0.e